# 11b — Figure 2/3 full-catalogue Monte Carlo refitting

This is a separate experiment. Notebook 11 and all its results remain unchanged. The model families and data selection are frozen at the completed Notebook 11 choices; **PySR is not re-run or reselected for each mock catalogue**.


## 1. What changes from Notebook 11?

Notebook 11 approximated coefficient uncertainty with a first-order Gaussian covariance. Here, each Monte Carlo realization draws a complete correlated mock data vector from the prepared covariance and refits the fixed quadratic or exponential family using generalized least squares (GLS). This is the catalogue-level workflow analogous to Notebook 07. Two centres are run independently: the fitted expression and the observed data.

For a shared realization of $r_d$, the supernova distances and their covariance are both rescaled coherently by $r_{d,0}/r_d$ and $(r_{d,0}/r_d)^2$, respectively. This is exact for the *prepared distance-space Gaussian model*; the original magnitude-to-distance covariance was already constructed with a first-order Jacobian in Notebook 08. Whitened sufficient statistics avoid storing a huge dense mock catalogue for every realization while giving the same GLS fit. The exponential basis uses a numerically checked 26th-order Taylor representation.


In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display, Image

PROJECT_ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                    if (p / 'results/08_fig23_data_preparation/fig2_pantheon.npz').is_file())
sys.path.insert(0, str(PROJECT_ROOT))
from src.fig23_catalog_bootstrap import run_catalog_bootstrap
PRIOR = (PROJECT_ROOT / 'results/11_fig23_mc_and_reconstruction' /
         'fig23_mc_20261010T044029855521Z_5cf83467')
OUTPUT_ROOT = PROJECT_ROOT / 'results/11b_fig23_catalog_bootstrap'
print('Frozen Notebook 11 result:', PRIOR)

## 2. Run or inspect

The default is inspection mode so opening the notebook in VS Code does not start a long experiment. Set `RUN_ANALYSIS=True` to create a **new timestamped folder**. To resume an interrupted run, set `RESUME_DIRECTORY` to its exact folder and keep all settings unchanged. The command-line runner `run_notebook11b.py` exposes the same settings.


In [ ]:
RUN_ANALYSIS = False
N_DRAWS = 20000  # per branch and per centre
BATCH_SIZE = 500
BASE_SEED = 20261014
RESUME_DIRECTORY = None

if RUN_ANALYSIS:
    EXPERIMENT_DIR, OVERVIEW = run_catalog_bootstrap(
        PROJECT_ROOT, PRIOR, n_draws=N_DRAWS, batch_size=BATCH_SIZE,
        base_seed=BASE_SEED, resume_directory=RESUME_DIRECTORY)
else:
    complete = sorted(p for p in OUTPUT_ROOT.glob('fig23_catalog_bootstrap_*')
                      if (p / 'experiment.json').is_file() and
                      json.loads((p / 'experiment.json').read_text()).get('status') == 'complete')
    if not complete:
        raise FileNotFoundError('No completed 11b run; set RUN_ANALYSIS=True.')
    EXPERIMENT_DIR = complete[-1]
    OVERVIEW = pd.read_csv(EXPERIMENT_DIR / 'overview.csv')
print('Experiment:', EXPERIMENT_DIR)
display(OVERVIEW)

## 3. Read the diagnostics and the method comparison

`all` retains finite draws, even if their $w(z)$ curve contains a pole on the plotted grid. `pole_free` also excludes pole-crossing draws; it does not additionally require a monotonically increasing distance. That condition is saved as a separate flag in the fit samples. Both are **pointwise** intervals, not simultaneous confidence bands. The comparison table uses the same 2.5–97.5% quantiles as Notebook 11. Differences therefore reflect catalogue refitting *and* the updated coherent $r_d$ covariance rescaling; they are not a pure isolation of nonlinearity.


In [ ]:
comparison = pd.read_csv(EXPERIMENT_DIR / 'comparison_to_linearized.csv')
display(comparison)
for scenario, branch in (('fig2_pantheon', 'independent'),
                         ('fig2_pantheon', 'refitted_exponential'),
                         ('fig3_desy5', 'independent')):
    print('\n', scenario, branch)
    details = json.loads((EXPERIMENT_DIR / scenario / branch / 'branch.json').read_text())
    print('Frozen structure:', details['source_expression'])
    display(Image(filename=str(EXPERIMENT_DIR / scenario / branch /
                               'best_fit_expression' / 'reconstruction.png')))
    display(Image(filename=str(EXPERIMENT_DIR / scenario / branch /
                               'observed_data' / 'reconstruction.png')))

## 4. Interpretation guardrails

A different $w(z)$ band does not by itself mean a different data fit. The two centring choices answer different questions: fitted-expression centring assumes that fixed family generated the data; observed-data centring preserves the actual residual pattern. Neither method re-runs symbolic structure discovery. Supernova data do extend below the lowest DESI BAO point, but $w(z)$ is derived from model derivatives rather than observed directly. Pointwise exclusion of $w=-1$ is not a global rejection of $\Lambda$CDM.
